# DeepSeek-V3 Multi-Head Latent Attention (MLA) - Memory Benchmark

This notebook demonstrates the 5x memory reduction achieved by DeepSeek MLA compared to standard Multi-Head Attention.

MLA compresses KV cache into a latent dimension while maintaining performance through Decoupled RoPE.

In [ ]:
# Install dependencies (run this first in Colab)
%pip install -q flax jax jaxlib matplotlib

## Step 1: DeepSeek MLA Implementation

In [ ]:
"""
DeepSeek-V3 Multi-Head Latent Attention (MLA) Implementation in JAX/Flax
"""

import jax
import jax.numpy as jnp
from flax import linen as nn
from typing import Optional, Tuple


class DeepSeekMLA(nn.Module):
    """
    DeepSeek Multi-Head Latent Attention (MLA) Module.
    
    Key innovation: Compresses KV cache into a latent dimension (kv_lora_rank)
    while using Decoupled RoPE to maintain attention quality.
    """
    num_heads: int
    head_dim: int
    kv_lora_rank: int
    q_lora_rank: int
    rope_dim: int
    dtype: jnp.dtype = jnp.float32
    
    def setup(self):
        """Initialize all projection matrices."""
        d_model = self.num_heads * self.head_dim
        
        # Compression matrices: project down to latent space
        self.kv_down_proj = nn.Dense(self.kv_lora_rank, use_bias=False, dtype=self.dtype, name='kv_down_proj')
        self.q_down_proj = nn.Dense(self.q_lora_rank, use_bias=False, dtype=self.dtype, name='q_down_proj')
        
        # Up-projection matrices: project back from latent space
        self.kv_up_proj = nn.Dense(self.num_heads * self.head_dim, use_bias=False, dtype=self.dtype, name='kv_up_proj')
        self.q_up_proj = nn.Dense(self.num_heads * self.head_dim, use_bias=False, dtype=self.dtype, name='q_up_proj')
        
        # RoPE projection: bypasses compression for rotary embeddings
        self.q_rope_proj = nn.Dense(self.num_heads * self.rope_dim, use_bias=False, dtype=self.dtype, name='q_rope_proj')
        self.k_rope_proj = nn.Dense(self.num_heads * self.rope_dim, use_bias=False, dtype=self.dtype, name='k_rope_proj')
        
        # Output projection
        self.out_proj = nn.Dense(d_model, use_bias=False, dtype=self.dtype, name='out_proj')
    
    def apply_rotary_pos_emb(self, x: jnp.ndarray, freqs: jnp.ndarray) -> jnp.ndarray:
        """Apply rotary position embeddings (RoPE)."""
        x_real, x_imag = jnp.split(x, 2, axis=-1)
        cos_freqs = jnp.cos(freqs)
        sin_freqs = jnp.sin(freqs)
        x_rotated_real = x_real * cos_freqs - x_imag * sin_freqs
        x_rotated_imag = x_real * sin_freqs + x_imag * cos_freqs
        return jnp.concatenate([x_rotated_real, x_rotated_imag], axis=-1)
    
    def get_rotary_freqs(self, seq_len: int) -> jnp.ndarray:
        """Generate rotary position embedding frequencies."""
        inv_freq = 1.0 / (10000 ** (jnp.arange(0, self.rope_dim, 2, dtype=self.dtype) / self.rope_dim))
        t = jnp.arange(seq_len, dtype=self.dtype)
        freqs = jnp.einsum('i,j->ij', t, inv_freq)
        freqs = jnp.expand_dims(jnp.expand_dims(freqs, 0), 0)
        return freqs
    
    def __call__(self, x: jnp.ndarray, cache: Optional[dict] = None, 
                 position_ids: Optional[jnp.ndarray] = None, training: bool = False) -> Tuple[jnp.ndarray, dict]:
        """Forward pass of MLA."""
        batch, seq_len, _ = x.shape
        
        # Step 1: Compress input to latent space
        kv_compressed = self.kv_down_proj(x)
        q_compressed = self.q_down_proj(x)
        
        # Step 2: Generate RoPE components (bypass compression)
        q_rope = self.q_rope_proj(x).reshape(batch, seq_len, self.num_heads, self.rope_dim)
        k_rope = self.k_rope_proj(x).reshape(batch, seq_len, self.num_heads, self.rope_dim)
        
        # Step 3: Apply rotary position embeddings
        if position_ids is None:
            position_ids = jnp.arange(seq_len)
        freqs = self.get_rotary_freqs(seq_len)
        if position_ids.shape[0] == seq_len:
            freqs = freqs[0, position_ids, :, :]
            freqs = jnp.expand_dims(freqs, 0)
        q_rope = self.apply_rotary_pos_emb(q_rope, freqs)
        k_rope = self.apply_rotary_pos_emb(k_rope, freqs)
        
        # Step 4: Up-project compressed vectors to full dimension (ON-THE-FLY, NOT stored in cache)
        kv_up = self.kv_up_proj(kv_compressed).reshape(batch, seq_len, self.num_heads, self.head_dim)
        q_up = self.q_up_proj(q_compressed).reshape(batch, seq_len, self.num_heads, self.head_dim)
        
        # Step 5: Combine content and RoPE components
        content_dim = self.head_dim - self.rope_dim
        q_content = q_up[:, :, :, :content_dim]
        k_content = kv_up[:, :, :, :content_dim]
        v_full = kv_up
        
        q_combined = jnp.concatenate([q_content, q_rope], axis=-1)
        k_combined = jnp.concatenate([k_content, k_rope], axis=-1)
        
        # Step 6: Compute attention
        q_combined = jnp.transpose(q_combined, (0, 2, 1, 3))
        k_combined = jnp.transpose(k_combined, (0, 2, 1, 3))
        v_full = jnp.transpose(v_full, (0, 2, 1, 3))
        
        # Handle cache for inference
        if cache is not None:
            k_cache_compressed = cache.get('k_compressed', None)
            v_cache_compressed = cache.get('v_compressed', None)
            k_rope_cache = cache.get('k_rope', None)
            
            if k_cache_compressed is not None and k_cache_compressed.shape[1] > 0:
                cache_len = k_cache_compressed.shape[1]
                # Up-project cached compressed vectors on-the-fly
                k_cache_up = self.kv_up_proj(k_cache_compressed).reshape(batch, cache_len, self.num_heads, self.head_dim)
                k_cache_content = k_cache_up[:, :, :, :content_dim]
                
                if k_rope_cache is not None:
                    k_rope_cached = k_rope_cache
                else:
                    cache_freqs = self.get_rotary_freqs(cache_len)
                    cache_freqs = cache_freqs[0, :cache_len, :, :]
                    cache_freqs = jnp.expand_dims(cache_freqs, 0)
                    k_rope_cached = jnp.zeros((batch, cache_len, self.num_heads, self.rope_dim), dtype=self.dtype)
                    k_rope_cached = self.apply_rotary_pos_emb(k_rope_cached, cache_freqs)
                
                k_cache_combined = jnp.concatenate([k_cache_content, k_rope_cached], axis=-1)
                k_cache_combined = jnp.transpose(k_cache_combined, (0, 2, 1, 3))
                k_combined = jnp.concatenate([k_cache_combined, k_combined], axis=2)
                
                v_cache_up = self.kv_up_proj(v_cache_compressed).reshape(batch, cache_len, self.num_heads, self.head_dim)
                v_cache_up = jnp.transpose(v_cache_up, (0, 2, 1, 3))
                v_full = jnp.concatenate([v_cache_up, v_full], axis=2)
        
        # Compute attention scores
        scale = 1.0 / jnp.sqrt(self.head_dim)
        attn_scores = jnp.einsum('bhqd,bhkd->bhqk', q_combined, k_combined) * scale
        
        if cache is None:
            mask = jnp.triu(jnp.ones((seq_len, seq_len), dtype=self.dtype), k=1) * -1e9
            attn_scores = attn_scores + mask[jnp.newaxis, jnp.newaxis, :, :]
        
        attn_probs = nn.softmax(attn_scores, axis=-1)
        attn_output = jnp.einsum('bhqk,bhvd->bhqd', attn_probs, v_full)
        
        attn_output = jnp.transpose(attn_output, (0, 2, 1, 3))
        attn_output = attn_output.reshape(batch, seq_len, self.num_heads * self.head_dim)
        output = self.out_proj(attn_output)
        
        # Update cache: ONLY store compressed vectors and RoPE components
        new_cache = {}
        if cache is not None:
            k_compressed_cached = cache.get('k_compressed', None)
            v_compressed_cached = cache.get('v_compressed', None)
            k_rope_cached = cache.get('k_rope', None)
            
            if k_compressed_cached is not None and k_compressed_cached.shape[1] > 0:
                new_cache['k_compressed'] = jnp.concatenate([k_compressed_cached, kv_compressed], axis=1)
                new_cache['v_compressed'] = jnp.concatenate([v_compressed_cached, kv_compressed], axis=1)
                if k_rope_cached is not None:
                    new_cache['k_rope'] = jnp.concatenate([k_rope_cached, k_rope], axis=1)
                else:
                    new_cache['k_rope'] = k_rope
            else:
                new_cache['k_compressed'] = kv_compressed
                new_cache['v_compressed'] = kv_compressed
                new_cache['k_rope'] = k_rope
        else:
            new_cache['k_compressed'] = kv_compressed
            new_cache['v_compressed'] = kv_compressed
            new_cache['k_rope'] = k_rope
        
        return output, new_cache
    
    def init_cache(self, batch: int, max_len: int) -> dict:
        """Initialize KV cache - stores ONLY compressed vectors, NOT up-projected matrices."""
        return {
            'k_compressed': jnp.zeros((batch, 0, self.kv_lora_rank), dtype=self.dtype),
            'v_compressed': jnp.zeros((batch, 0, self.kv_lora_rank), dtype=self.dtype),
            'k_rope': jnp.zeros((batch, 0, self.num_heads, self.rope_dim), dtype=self.dtype),
        }

print("DeepSeek MLA implementation loaded")

## Step 2: Standard MHA Baseline

In [ ]:
class StandardMHA(nn.Module):
    """Standard Multi-Head Attention (Vanilla MHA) for baseline comparison."""
    num_heads: int
    head_dim: int
    dtype: jnp.dtype = jnp.float32
    
    def setup(self):
        d_model = self.num_heads * self.head_dim
        self.q_proj = nn.Dense(d_model, use_bias=False, dtype=self.dtype)
        self.k_proj = nn.Dense(d_model, use_bias=False, dtype=self.dtype)
        self.v_proj = nn.Dense(d_model, use_bias=False, dtype=self.dtype)
        self.out_proj = nn.Dense(d_model, use_bias=False, dtype=self.dtype)
    
    def __call__(self, x: jnp.ndarray, cache: Optional[dict] = None, training: bool = False) -> Tuple[jnp.ndarray, dict]:
        batch, seq_len, _ = x.shape
        q = self.q_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        v = self.v_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        
        q = jnp.transpose(q, (0, 2, 1, 3))
        k = jnp.transpose(k, (0, 2, 1, 3))
        v = jnp.transpose(v, (0, 2, 1, 3))
        
        if cache is not None:
            k_cache = cache.get('k', None)
            v_cache = cache.get('v', None)
            if k_cache is not None:
                k = jnp.concatenate([k_cache, k], axis=2)
                v = jnp.concatenate([v_cache, v], axis=2)
        
        scale = 1.0 / jnp.sqrt(self.head_dim)
        attn_scores = jnp.einsum('bhqd,bhkd->bhqk', q, k) * scale
        
        if cache is None:
            mask = jnp.triu(jnp.ones((seq_len, seq_len), dtype=self.dtype), k=1) * -1e9
            attn_scores = attn_scores + mask[jnp.newaxis, jnp.newaxis, :, :]
        
        attn_probs = nn.softmax(attn_scores, axis=-1)
        attn_output = jnp.einsum('bhqk,bhvd->bhqd', attn_probs, v)
        
        attn_output = jnp.transpose(attn_output, (0, 2, 1, 3))
        attn_output = attn_output.reshape(batch, seq_len, self.num_heads * self.head_dim)
        output = self.out_proj(attn_output)
        
        new_cache = {}
        if cache is not None:
            k_cached = cache.get('k', None)
            v_cached = cache.get('v', None)
            if k_cached is not None:
                new_cache['k'] = jnp.concatenate([k_cached, k], axis=2)
                new_cache['v'] = jnp.concatenate([v_cached, v], axis=2)
            else:
                new_cache['k'] = k
                new_cache['v'] = v
        else:
            new_cache['k'] = k
            new_cache['v'] = v
        
        return output, new_cache
    
    def init_cache(self, batch: int, max_len: int) -> dict:
        """Initialize cache storing full KV matrices."""
        return {
            'k': jnp.zeros((batch, self.num_heads, 0, self.head_dim), dtype=self.dtype),
            'v': jnp.zeros((batch, self.num_heads, 0, self.head_dim), dtype=self.dtype),
        }

print("Standard MHA implementation loaded")

## Step 3: Memory Comparison Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def calculate_memory_footprint_mha(seq_len: int, num_heads: int, head_dim: int, batch: int = 1, dtype_bytes: int = 2) -> float:
    """Calculate KV cache memory footprint for Standard MHA."""
    memory_bytes = 2 * seq_len * num_heads * head_dim * batch * dtype_bytes
    return memory_bytes / (1024 * 1024)


def calculate_memory_footprint_mla(seq_len: int, kv_lora_rank: int, num_heads: int, rope_dim: int, 
                                    batch: int = 1, dtype_bytes: int = 2) -> float:
    """Calculate KV cache memory footprint for DeepSeek MLA."""
    memory_bytes = seq_len * (kv_lora_rank + num_heads * rope_dim) * batch * dtype_bytes
    return memory_bytes / (1024 * 1024)


def compare_memory(seq_lens: list, num_heads: int = 32, head_dim: int = 128, 
                   kv_lora_rank: int = 64, rope_dim: int = 64, batch: int = 1, dtype_bytes: int = 2):
    """Compare memory footprints between Standard MHA and DeepSeek MLA."""
    print("=" * 80)
    print("Memory Footprint Comparison: Standard MHA vs DeepSeek MLA")
    print("=" * 80)
    print(f"\nConfiguration:")
    print(f"  Num Heads: {num_heads}")
    print(f"  Head Dim: {head_dim}")
    print(f"  KV LoRA Rank (MLA): {kv_lora_rank}")
    print(f"  RoPE Dim (MLA): {rope_dim}")
    print(f"  Batch Size: {batch}")
    print(f"  Dtype: {dtype_bytes * 8}-bit")
    print("\n" + "-" * 80)
    print(f"{'Seq Len':<12} {'MHA (MB)':<15} {'MLA (MB)':<15} {'Reduction':<15}")
    print("-" * 80)
    
    mha_memories = []
    mla_memories = []
    reductions = []
    
    for seq_len in seq_lens:
        mha_mem = calculate_memory_footprint_mha(seq_len, num_heads, head_dim, batch, dtype_bytes)
        mla_mem = calculate_memory_footprint_mla(seq_len, kv_lora_rank, num_heads, rope_dim, batch, dtype_bytes)
        reduction = mha_mem / mla_mem if mla_mem > 0 else 0
        
        mha_memories.append(mha_mem)
        mla_memories.append(mla_mem)
        reductions.append(reduction)
        
        print(f"{seq_len:<12} {mha_mem:<15.2f} {mla_mem:<15.2f} {reduction:.2f}x")
    
    print("=" * 80)
    return mha_memories, mla_memories, reductions


def visualize_comparison(seq_lens: list, mha_memories: list, mla_memories: list):
    """Generate matplotlib visualization comparing memory usage."""
    plt.figure(figsize=(12, 8))
    
    plt.plot(seq_lens, mha_memories, 'b-o', label='Standard MHA', linewidth=2, markersize=6)
    plt.plot(seq_lens, mla_memories, 'r-s', label='DeepSeek MLA (JAX)', linewidth=2, markersize=6)
    
    plt.xlabel('Sequence Length', fontsize=14, fontweight='bold')
    plt.ylabel('KV Cache Memory Usage (MB)', fontsize=14, fontweight='bold')
    plt.title('Memory Footprint Comparison: Standard MHA vs DeepSeek MLA', 
              fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xscale('log', base=2)
    plt.yscale('log')
    
    if len(seq_lens) > 0:
        max_idx = len(seq_lens) - 1
        reduction = mha_memories[max_idx] / mla_memories[max_idx] if mla_memories[max_idx] > 0 else 0
        plt.annotate(
            f'{reduction:.1f}x reduction\nat {seq_lens[max_idx]} tokens',
            xy=(seq_lens[max_idx], mla_memories[max_idx]),
            xytext=(20, 20),
            textcoords='offset points',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'),
            fontsize=10
        )
    
    plt.tight_layout()
    plt.savefig('mla_memory_comparison.png', dpi=150, bbox_inches='tight')
    print("\nVisualization saved as 'mla_memory_comparison.png'")
    plt.show()

print("Memory comparison functions loaded")

## Step 4: Run Benchmark

In [ ]:
# Check TPU availability
print("Checking JAX devices...")
print(f"Available devices: {jax.devices()}")
print(f"Device count: {len(jax.devices())}")
print(f"Default backend: {jax.default_backend()}")
print()

# Configuration matching DeepSeek-V3 style architecture
NUM_HEADS = 32
HEAD_DIM = 128
KV_LORA_RANK = 64  # Compressed dimension
Q_LORA_RANK = 64
ROPE_DIM = 64

# Sequence lengths to test (powers of 2 from 1024 to 32768)
SEQ_LENS = [1024, 2048, 4096, 8192, 16384, 32768]

# Run comparison
mha_memories, mla_memories, reductions = compare_memory(
    SEQ_LENS,
    num_heads=NUM_HEADS,
    head_dim=HEAD_DIM,
    kv_lora_rank=KV_LORA_RANK,
    rope_dim=ROPE_DIM,
    batch=1,
    dtype_bytes=2  # bfloat16/float16
)

# Visualize
visualize_comparison(SEQ_LENS, mha_memories, mla_memories)

# Print summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
avg_reduction = np.mean(reductions)
print(f"Average memory reduction: {avg_reduction:.2f}x")
print(f"Max sequence length tested: {max(SEQ_LENS)}")
print(f"Memory savings at max length: {mha_memories[-1] - mla_memories[-1]:.2f} MB")
print("=" * 80)